In [1]:
from pathlib import Path
import tensorflow as tf

I0000 00:00:1774883556.818549    6836 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774883557.768731    6836 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774883561.267864    6836 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
from pathlib import Path
import tensorflow as tf

# --- Determine project root ---
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

# --- Paths to data ---
wikiart_path = project_root / "data"
train_dir = wikiart_path / "train"
val_dir = wikiart_path / "validation"
test_dir = wikiart_path / "test"

# --- Check paths exist ---
for path in [train_dir, val_dir, test_dir]:
    if not path.exists():
        print(f"Warning: {path} does not exist!")

print(f"Project root: {project_root}")
print(f"WikiArt path: {wikiart_path}")

# --- Dataset parameters ---
BATCH_SIZE = 32
IMG_SIZE = (224, 224)  # Resize images for model input

# --- Load datasets ---
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

print("Datasets loaded successfully!")
print(f"Train batches: {len(train_ds)}, Validation batches: {len(val_ds)}, Test batches: {len(test_ds)}")

Project root: /teamspace/studios/this_studio/DeepLearning-NOVAIMS2026
WikiArt path: /teamspace/studios/this_studio/DeepLearning-NOVAIMS2026/data
Found 9326 files belonging to 23 classes.


E0000 00:00:1774883565.731700    6836 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1774883566.019822    6836 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Found 1992 files belonging to 23 classes.
Found 2022 files belonging to 23 classes.
Datasets loaded successfully!
Train batches: 292, Validation batches: 63, Test batches: 64


In [3]:
#!pip install transformers datasets torch torchvision pillow

In [4]:
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image

# --- Load processor ---
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')

# --- Create a PyTorch Dataset from folders ---
class ImageFolderDataset(Dataset):
    def __init__(self, path, processor, transform=None):
        self.path = Path(path)
        self.processor = processor
        self.transform = transform
        self.samples = []
        self.classes = sorted([d.name for d in path.iterdir() if d.is_dir()])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        for cls in self.classes:
            cls_dir = path / cls
            for img_path in cls_dir.glob('*'):
                self.samples.append((img_path, self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        # ViT processor converts PIL image to tensor and normalizes
        encoding = self.processor(images=image, return_tensors="pt")
        pixel_values = encoding['pixel_values'].squeeze()  # remove batch dim
        return pixel_values, label

# --- Define transform (optional) ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ViT expects 224x224
])

# --- Create datasets ---
train_dataset = ImageFolderDataset(train_dir, processor, transform)
val_dataset = ImageFolderDataset(val_dir, processor, transform)

# --- DataLoaders ---
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [5]:
from transformers import ViTForImageClassification

num_labels = len(train_dataset.classes)
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224',
    num_labels=num_labels,
    id2label={i: cls for i, cls in enumerate(train_dataset.classes)},
    label2id={cls: i for i, cls in enumerate(train_dataset.classes)},
    ignore_mismatched_sizes=True  # THIS fixes the issue
)

You passed `num_labels=23` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                           
------------------+----------+-------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([23])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([23, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [6]:
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from sklearn.metrics import f1_score
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)
loss_fn = CrossEntropyLoss()
epochs = 3  # adjust as needed

for epoch in range(epochs):
    start_time = time.time()
    
    # --- Training ---
    model.train()
    train_loss = 0
    train_preds = []
    train_labels = []

    for pixel_values, labels in train_loader:
        pixel_values, labels = pixel_values.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values)
        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    train_loss /= len(train_loader)
    train_f1 = f1_score(train_labels, train_preds, average='macro')

    # --- Validation ---
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for pixel_values, labels in val_loader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            outputs = model(pixel_values=pixel_values)
            loss = loss_fn(outputs.logits, labels)

            val_loss += loss.item()
            val_preds.extend(outputs.logits.argmax(-1).cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_f1 = f1_score(val_labels, val_preds, average='macro')

    # --- Epoch summary ---
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{epochs} - time: {epoch_time:.1f}s - "
          f"loss: {train_loss:.4f} - f1: {train_f1:.4f} - "
          f"val_loss: {val_loss:.4f} - val_f1: {val_f1:.4f}")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 302: Error loading CUDA libraries. GPU will not be used. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Epoch 1/3 - time: 4392.7s - loss: 1.2113 - f1: 0.6372 - val_loss: 0.6343 - val_f1: 0.8103
Epoch 2/3 - time: 4238.1s - loss: 0.2831 - f1: 0.9198 - val_loss: 0.5660 - val_f1: 0.8198
Epoch 3/3 - time: 4403.0s - loss: 0.0473 - f1: 0.9926 - val_loss: 0.4819 - val_f1: 0.8487


In [7]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        pixel_values, labels = batch
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)

        outputs = model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

val_f1 = f1_score(all_labels, all_preds, average='macro')
print(f"Validation F1-score: {val_f1:.4f}")

Validation F1-score: 0.8487


In [11]:
from sklearn.metrics import classification_report

print(classification_report(
    all_labels,
    all_preds,
    target_names=train_dataset.classes
))

                       precision    recall  f1-score   support

       Albrecht_Durer       0.86      0.91      0.88        87
      Boris_Kustodiev       0.77      0.80      0.79        66
     Camille_Pissarro       0.76      0.84      0.80        93
        Childe_Hassam       0.88      0.88      0.88        57
         Claude_Monet       0.89      0.89      0.89       140
          Edgar_Degas       0.82      0.77      0.79        64
        Eugene_Boudin       0.98      0.91      0.95        58
         Gustave_Dore       0.91      0.95      0.93        79
           Ilya_Repin       0.91      0.77      0.83        56
      Ivan_Aivazovsky       0.88      0.98      0.93        60
        Ivan_Shishkin       0.97      0.72      0.83        54
  John_Singer_Sargent       0.84      0.91      0.88        82
         Marc_Chagall       0.94      0.78      0.85        80
      Martiros_Saryan       0.76      0.90      0.82        60
     Nicholas_Roerich       0.92      0.90      0.91  

In [9]:
test_dataset = ImageFolderDataset(test_dir, processor, transform)
from torch.utils.data import DataLoader

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

In [10]:
from sklearn.metrics import f1_score

model.eval()

test_preds = []
test_labels = []

with torch.no_grad():
    for pixel_values, labels in test_loader:
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)

        outputs = model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(-1)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

# Compute F1 score
test_f1 = f1_score(test_labels, test_preds, average='macro')

print(f"Test F1-score: {test_f1:.4f}")

Test F1-score: 0.8422


In [12]:
from sklearn.metrics import classification_report

print(classification_report(
    test_labels,
    test_preds,
    target_names=train_dataset.classes
))

                       precision    recall  f1-score   support

       Albrecht_Durer       0.80      0.94      0.86        87
      Boris_Kustodiev       0.81      0.76      0.79        68
     Camille_Pissarro       0.73      0.86      0.79        94
        Childe_Hassam       0.83      0.73      0.77        59
         Claude_Monet       0.91      0.85      0.88       141
          Edgar_Degas       0.86      0.78      0.82        65
        Eugene_Boudin       0.91      0.86      0.89        59
         Gustave_Dore       0.93      0.96      0.94        80
           Ilya_Repin       0.83      0.66      0.73        58
      Ivan_Aivazovsky       0.95      0.92      0.93        62
        Ivan_Shishkin       0.94      0.80      0.87        56
  John_Singer_Sargent       0.82      0.89      0.86        83
         Marc_Chagall       1.00      0.78      0.88        81
      Martiros_Saryan       0.76      0.84      0.80        61
     Nicholas_Roerich       0.96      0.88      0.92  